In [1]:
import os
import time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/data', force_remount=True)

ROOT_DIRECTORY = 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista'
SPLITS_CSV_PATH = ROOT_DIRECTORY + '/k-fold_split/typography_dataset_splits.csv'

Mounted at /content/data


In [36]:
ROOT_DIRECTORY = '../../data'
SPLITS_CSV_PATH = ROOT_DIRECTORY + '/k-fold_split/typography_dataset_splits_local.csv'

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [4]:
class TypographyDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'image_path']
        label = self.dataframe.loc[idx, 'label_encoded']
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            image = Image.new('RGB', (224, 224))
            
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

In [5]:
def load_fold_from_csv(csv_path, target_fold_idx, label_encoder):
    df = pd.read_csv(csv_path)
    df['label_encoded'] = label_encoder.transform(df['label'])
    
    fold_df = df[df['fold'] == target_fold_idx]
    train_df = fold_df[fold_df['split_type'] == 'train']
    val_df = fold_df[fold_df['split_type'] == 'val']
    
    return train_df, val_df

In [6]:
# ==========================================
# 2. DEFINIÇÃO DAS TRANSFORMAÇÕES (ATUALIZADO)
# ==========================================
train_transform = transforms.Compose([
    transforms.Resize((256, 256)), 
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15), 
    transforms.RandomResizedCrop(size=224, scale=(0.8, 1.0)), 
    transforms.ColorJitter(brightness=0.3, contrast=0.3), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [7]:
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [8]:
# ==========================================
# 3. LOOP DE TREINAMENTO DO FOLD
# ==========================================
def train_resnet_fold(train_loader, val_loader, num_classes=4, epochs=25): # Reduzimos para 25
    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    num_ftrs = model.fc.in_features
    
    # 1. SUBSTITUIÇÃO: Adicionamos o Dropout antes da decisão final
    model.fc = nn.Sequential(
        nn.Dropout(p=0.5), # Desliga 50% dos neurônios aleatoriamente no treino
        nn.Linear(num_ftrs, num_classes)
    )
    
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    
    # 2. SUBSTITUIÇÃO: Adicionamos o weight_decay (Regularização L2)
    optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
    
    # Barra de progresso envolvendo o range de épocas
    epoch_iterator = tqdm(range(epochs), desc="Treinando Épocas", unit="época")
    
    for epoch in epoch_iterator:
        model.train()
        running_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            
        # Calcula a loss média da época para mostrar na barra de progresso
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_iterator.set_postfix({"Loss": f"{epoch_loss:.4f}"})
            
    # Fase de Validação final do fold
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    return all_labels, all_preds

In [9]:
def run_cv_resnet_augmented(csv_path, num_folds=5, batch_size=32, epochs=15):
    print(f"Iniciando Validação Cruzada com ResNet-18 (COM AUGMENTATION) por {epochs} épocas...")
    
    classes = sorted(['grotesco', 'serifado', 'escritural', 'fantasia'])
    label_encoder = LabelEncoder()
    label_encoder.fit(classes)
    
    fold_metrics = []
    
    for fold_idx in range(num_folds):
        print(f"\n{'='*40}")
        print(f"Processando Fold {fold_idx + 1}/{num_folds} (ResNet-18 Augmentation)")
        print(f"{'='*40}")
        
        train_df, val_df = load_fold_from_csv(csv_path, fold_idx, label_encoder)
        
        # APLICAÇÃO DOS TRANSFORMS DIFERENTES AQUI
        train_dataset = TypographyDataset(train_df, transform=train_transform)
        val_dataset = TypographyDataset(val_df, transform=val_transform)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        start_time = time.time()
        
        y_true, y_pred = train_resnet_fold(train_loader, val_loader, num_classes=4, epochs=epochs)
        
        training_time = time.time() - start_time
        fold_accuracy = accuracy_score(y_true, y_pred)
        
        print(f"Fold {fold_idx + 1} Acurácia: {fold_accuracy:.4f} (Tempo: {training_time:.2f}s)")
        print(classification_report(y_true, y_pred, labels=[0, 1, 2, 3], target_names=classes, zero_division=0))
        
        fold_metrics.append(fold_accuracy)
        
    avg_accuracy = np.mean(fold_metrics)
    std_accuracy = np.std(fold_metrics)
    
    print(f"\n{'='*40}")
    print("RESULTADO FINAL CROSS-VALIDATION - RESNET-18 (COM AUGMENTATION)")
    print(f"Acurácia Média: {avg_accuracy:.4f} ± {std_accuracy:.4f}")
    print(f"{'='*40}\n")

In [10]:

if __name__ == "__main__":
    if os.path.exists(SPLITS_CSV_PATH):
        # Como temos Data Augmentation, a rede precisa de mais épocas para aprender a variação. 
        # 15 épocas é um ponto de partida excelente.
        run_cv_resnet_augmented(SPLITS_CSV_PATH, num_folds=5, batch_size=32, epochs=25)
    else:
        print("Erro: CSV não encontrado.")

Iniciando Validação Cruzada com ResNet-18 (COM AUGMENTATION) por 25 épocas...

Processando Fold 1/5 (ResNet-18 Augmentation)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 180MB/s] 
Treinando Épocas: 100%|██████████| 25/25 [09:07<00:00, 21.89s/época, Loss=0.0604]  


Fold 1 Acurácia: 0.5660 (Tempo: 613.32s)
              precision    recall  f1-score   support

  escritural       0.71      0.87      0.78        94
    fantasia       0.00      0.00      0.00        11
    grotesco       0.31      0.26      0.28        54
    serifado       0.48      0.45      0.47        53

    accuracy                           0.57       212
   macro avg       0.38      0.40      0.38       212
weighted avg       0.52      0.57      0.54       212


Processando Fold 2/5 (ResNet-18 Augmentation)


Treinando Épocas: 100%|██████████| 25/25 [04:25<00:00, 10.63s/época, Loss=0.0635]


Fold 2 Acurácia: 0.6278 (Tempo: 267.24s)
              precision    recall  f1-score   support

  escritural       0.57      0.86      0.68        49
    fantasia       0.83      0.24      0.37        21
    grotesco       0.69      0.76      0.72        90
    serifado       0.57      0.40      0.47        63

    accuracy                           0.63       223
   macro avg       0.66      0.56      0.56       223
weighted avg       0.64      0.63      0.61       223


Processando Fold 3/5 (ResNet-18 Augmentation)


...

Fold 3 Acurácia: 0.7784 (Tempo: 275.66s)
              precision    recall  f1-score   support

  escritural       0.87      0.94      0.90        89
    fantasia       0.00      0.00      0.00         8
    grotesco       0.90      0.68      0.77        78
    serifado       0.37      0.74      0.49        19

    accuracy                           0.78       194
   macro avg       0.53      0.59      0.54       194
weighted avg       0.79      0.78      0.77       194


Processando Fold 4/5 (ResNet-18 Augmentation)


Treinando Épocas: 100%|██████████| 25/25 [04:14<00:00, 10.16s/época, Loss=0.0882]


Fold 4 Acurácia: 0.7186 (Tempo: 256.13s)
              precision    recall  f1-score   support

  escritural       0.50      0.93      0.65        57
    fantasia       0.33      0.08      0.12        13
    grotesco       0.95      0.83      0.89       150
    serifado       0.43      0.23      0.30        43

    accuracy                           0.72       263
   macro avg       0.55      0.52      0.49       263
weighted avg       0.74      0.72      0.70       263


Processando Fold 5/5 (ResNet-18 Augmentation)


Treinando Épocas: 100%|██████████| 25/25 [04:19<00:00, 10.39s/época, Loss=0.0468]


Fold 5 Acurácia: 0.8368 (Tempo: 261.48s)
              precision    recall  f1-score   support

  escritural       0.82      0.88      0.84        96
    fantasia       0.00      0.00      0.00         0
    grotesco       0.85      0.86      0.86       101
    serifado       0.88      0.69      0.77        42

    accuracy                           0.84       239
   macro avg       0.64      0.61      0.62       239
weighted avg       0.84      0.84      0.84       239


RESULTADO FINAL CROSS-VALIDATION - RESNET-18 (COM AUGMENTATION)
Acurácia Média: 0.7055 ± 0.0982

